# Dataset ransomware — pipeline 2026

Tre fonti, cinque stadi. Ogni stadio è uno script autonomo che scrive un proprio file: niente riscritture in-place, rieseguibile da capo, ogni passo ispezionabile. I fetch hanno checkpoint e si possono lanciare in `screen`.

| stadio | script | output |
|---|---|---|
| 1a | `fetch_ransomfeed.py` | `raw/ransomfeed_raw.json` |
| 1b | `fetch_ransomlive.py` | `raw/ransomlive_raw.json` |
| 1c | `fetch_ransomlook.py` | `raw/ransomlook_raw.json` |
| 3 | `stage3_unify.py` | `build/unified.json` — un record per riga, tre fonti |
| 4 | `stage4_events.py` | `build/events.json` — un evento per riga, con provenienza |

Moduli condivisi: `normalize.py` (ISO 3166-1, gang), `ransomlook_clean.py` (titoli RansomLook), `gang_aliases_manual.json` (revisione manuale degli alias).

**Non ancora in pipeline**: consolidamento della tassonomia dei settori, recupero dei valori mancanti.

## Stadio 1 — fetch

Lanciarli in `screen` (durano da minuti a un'ora):
```bash
screen -dmS rf  bash -c 'python3 -u fetch_ransomfeed.py 2>&1 | tee logs/ransomfeed.log'
screen -dmS rl  bash -c 'RANSOMLIVE_API_KEY=... python3 -u fetch_ransomlive.py 2>&1 | tee logs/ransomlive.log'
screen -dmS rlk bash -c 'python3 -u fetch_ransomlook.py 2>&1 | tee logs/ransomlook.log'
```

**Limiti delle API, misurati e aggirati**

- *ransomfeed*: `offset` è un limite, non un cursore, e **sopra 1000 il server torna al default di 100**. Le coppie gang-anno che restituiscono esattamente 1000 record sono troncate e vengono rispezzate per paese (`/gang/{g}/date/{y}/country/{c}`). Copertura verificata: tutti i record esposti dall'API; `/stats.total` include voci non accessibili con alcun filtro.
- *ransomware.live*: `year` ora richiede `month`; lo script precedente riceveva 400 su ogni chiamata. Il vecchio ciclo per settore perdeva tutte le vittime con settore nullo: senza filtro i record passano da 16.767 a 32.111.
- *ransomlook*: nessuna chiave, contenuti CC BY 4.0; copertura 100% rispetto a `/api/stats`. **Nessun paese né settore**: serve a copertura e corroborazione.

In [ ]:
import json, pandas as pd
for f in ("ransomfeed","ransomlive","ransomlook"):
    d=json.load(open(f"raw/{f}_raw.json",encoding="utf-8"))
    extra = {k:d.get(k) for k in ("api_total","n_gangs") if d.get(k) is not None}
    print(f"{f:11} {d['n_records']:6d} record | fetched_at {d['fetched_at']} | fallimenti {len(d.get('failures',[]))}", extra)

## Stadio 2b — tassonomia dei settori

`build_sector_crosswalk.py` → `sector_crosswalk.json`.

**Target**: la tassonomia di ransomware.live (12 classi + `Other`): esterna, già in uso, copre la fonte più ricca, e le sue etichette entrano senza conversione.

**Le 175 etichette di ransomfeed** si mappano in due modi: `empirical` — sugli eventi etichettati da entrambe le fonti si prende la classe ransomware.live più frequente, se supporto ≥ 20 e purezza ≥ 0,5; `manual` — tutte le altre, decise a mano con il crosstab come riferimento. Ogni voce del file registra metodo, supporto e purezza.

Il crosswalk si costruisce su `build/unified.json` e `build/record_to_event.json`: al primo giro servono gli stadi 3 e 4 già eseguiti una volta; poi si rilanciano.

In [ ]:
%run build_sector_crosswalk.py

## Stadio 3 — unificazione

Un record per riga. Per ciascuno: vittima pulita, dominio, chiavi di identità (`vkey` dal nome, `dkey` dal dominio), gang canonica, paese ISO.

- **Titoli RansomLook**: il `post_title` è testo grezzo del leak site (`"x.com | 8TB documents"`, `"PART 2"`, `"(UPDATE)"`, bandiere-emoji). `clean_title` estrae nome, dominio e indizio di paese.
- **Gang**: fusioni conservative — stessa chiave, suffissi di servizio (`clop torrents`→`clop`), cifre finali solo se sono un artefatto di RansomLook (`killsec3`→`killsec`). Nessuna fusione per somiglianza: le coppie simili vanno in `gang_aliases_manual.json`, dove `lockbit3`/`lockbit5` o `blackmatter`/`blackwater` restano distinte.
- **Paesi**: 366 varianti grezze → ISO alpha-2, zero irrisolte. `NA` è la Namibia, non "non disponibile".

**Correzione delle chiavi di identità (v2).** La validazione delle regole ha mostrato chiavi condivise da vittime diverse: `https` (75 eventi: nome della vittima scritto come URL), `gov`/`gob` (`gov.br` ridotto al suffisso pubblico), `zoominfo` (43 eventi: alcune fonti riportano l'aggregatore al posto del sito). Correzioni:
- il nome-URL è ridotto all'host prima di calcolare la chiave;
- il dominio è ridotto al dominio registrabile con `tldextract` e la lista dei suffissi pubblici inclusa nel pacchetto (nessun download, risultato riproducibile): `gov.br` non è un'organizzazione;
- un dominio non valido (troncato `foo.co...`, `.onion`, testo libero) o di un aggregatore (`zoominfo.com`, `dnb.com`, `example.com`…) diventa null;
- chiavi generiche (`https`, `gov`, `test`, `k12`…) scartate.

Effetto: eventi da 37.700 a 37.650 (v2). Le correzioni successive emerse dall'audit (sezione *Audit della deduplicazione*) portano al totale finale di **37.385 eventi**. I risultati del sito già calcolati sono stati rimappati sui nuovi `event_id` tramite gli ID dei record grezzi (`build/event_id_map_v1_v2.json`), con verifica che il dominio coincida.

In [ ]:
%run stage3_unify.py

## Stadio 4 — eventi

Un **evento** è un attacco rivendicato: stessa vittima, stessa gang, date entro `WINDOW` giorni (default 3).

- L'identità della vittima unisce nome e dominio (union-find), a parità di gang.
- Le date di **RansomLook** prima del 2022 sono ricostruite a posteriori (scarti mediani di ~3 mesi): i suoi post **corroborano** l'evento più vicino delle fonti ricche invece di generarne di nuovi. Solo i post senza alcun riscontro diventano eventi propri.
- **Attributi con provenienza**: `country_src` / `sector_src` ∈ `observed`, `inherited_victim` (paese e settore descrivono la vittima, non l'attacco: si ereditano da altri eventi della stessa vittima), `title_hint`, `cctld`, `missing`.
- `country_conflict` segna gli eventi in cui le fonti riportano paesi diversi (tipicamente sede contro filiale). I conflitti di settore invece sono per lo più **tassonomie diverse** (175 etichette contro 14) e spariranno con il consolidamento.

In [ ]:
%run stage4_events.py

## Stadio 5c — paese dal sito della vittima (archive.org)

Dopo lo stadio 4 restano **6.579 eventi senza paese** (17,6%; numeri della build finale). Per quelli con un dominio si legge il sito dell'organizzazione stessa: è la fonte primaria, non un altro aggregatore.

`site_enrich.py`:
1. **Snapshot precedente all'attacco.** API `wayback/available` (1–2 s), accettata solo se lo snapshot è *strettamente* anteriore alla data dell'evento; altrimenti CDX con finestra di 3 anni. Motivo: dopo l'attacco il sito può essere defacciato, offline o venduto; lo snapshot è anche fisso, quindi riproducibile. Solo in assenza di snapshot si usa il sito attuale (origine `live_*`, tenuta distinta).
2. Home page e, se mancano segnali forti, una pagina contatti/impressum dello stesso snapshot.
3. Segnali di paese, registrati separatamente: **schema** (`addressCountry` schema.org), **vat** (P.IVA, USt-IdNr, SIRET, CNPJ, ABN, Companies House, KvK…), **address** (formati postali inequivocabili: stato USA + ZIP, CAP canadese, postcode UK), **phone** (`phonenumbers`; numeri verdi e non geografici esclusi perché non indicano un paese).
4. Decisione conservativa: `strong` se schema/vat/address concordano; altrimenti `phone` se tutti i telefoni indicano lo stesso paese (`phone_majority` ≥ 80%); in ogni altro caso nessuna risposta.

Scelte emerse dai test:
- il CDX risponde in 40–80 s: con timeout di 25 s il 73% delle pagine ripiegava sul sito attuale. Corretto con l'API availability + CDX a 120 s;
- 1.055 eventi hanno nel campo `domain` testo che non è un dominio (troncati `foo.co...`, `.onion`, frasi del leak site): scartati senza richieste di rete;
- controllo DNS prima del sito attuale, per non sprecare tentativi su domini morti;
- secondo passaggio a 4 worker sugli eventi in cui archive.org non aveva risposto per sovraccarico (snapshot storici da 716 a 1.202).

**Validazione** (`site_evaluate.py`): 600 eventi con paese noto e unanime, etichetta nascosta.
Risposta nel 52,0% dei casi, **precisione 95,8%** (`strong` 96,3% n=218; `phone` 94,6% n=92). Archive 96,0%, sito attuale 94,3%. Nessuno snapshot successivo all'attacco (0/487). Baseline "sempre US" sullo stesso campione: 48,2%; precisione delle risposte US 98,3% (n=180). Punto debole: GB 75% (n=16), tipicamente aziende statunitensi con filiale britannica.

**Applicazione** (build finale): 678 paesi (541 da snapshot, 137 dal sito attuale); paese mancante da 17,6% a 15,8%.
Provenienza: `country_src` = `site_archive` / `site_live`, più `site_rule` e `site_snapshot`.

In [ ]:
%run site_evaluate.py

## Stadio 5d — settore dal testo del sito (LLM senza ricerca)

`site_sector.py`: l'estratto della pagina salvato da `site_enrich.py` (titolo, meta description, intestazioni, primi 1.500 caratteri) è classificato in una delle 13 classi, con `unknown` ammesso. Nessuna ricerca web: il testo è già la fonte primaria, e una chiamata costa ~760 token contro 20.000–200.000 con la ricerca.

**Validazione** (`site_sector_evaluate.py`, `gpt-oss-120b`, 300 eventi a settore osservato). Si separano gli eventi in cui le fonti concordano (etichetta affidabile) da quelli in conflitto (corretto se coincide con uno dei candidati). Riferimento: le fonti stesse concordano solo nel ~60% dei casi (κ = 0,54).

| soglia | copertura | acc. (fonti concordi, n=198) | acc. su candidati (tutti) |
|---|---|---|---|
| — | 90% | 78,3% | 78,1% |
| **≥ 0,90** | 78% | **80,8%** | 80,9% |
| ≥ 0,95 | 65% | 85,5% | — |

Soglia scelta: **0,90**. Errori tra classi vicine (Manufacturing ↔ Professional Services, Agriculture ↔ Manufacturing).

**Applicazione: `gpt-oss-120b` in locale.** Il piano gratuito di Groq limita il modello a 200.000 token al giorno, quindi l'applicazione avrebbe richiesto giorni. Lo stesso modello (pesi pubblici `openai/gpt-oss-120b`, MXFP4) è stato eseguito in locale con `vllm` 0.30.0 su 4 GPU A100 (tensor parallel 4, `temperature=0`, `reasoning_effort=low`). Questo dà riproducibilità senza dipendere da un servizio esterno.

Per non mescolare backend, **validazione e applicazione sono state ripetute interamente in locale** sugli stessi 300 eventi:
- accordo con le risposte Groq sugli stessi eventi: **97,3%**;
- a soglia 0,90: accuratezza **82,3%** sugli eventi con fonti concordi, copertura 82,8%; 82,5% contando come corretti i candidati, su tutti gli eventi.

Applicazione agli eventi senza settore con testo del sito: build finale **289 settori accettati**. Settore mancante da 7,4% a **6,6%**. Le corse Groq parziali (`gpt-oss-120b` e `gpt-oss-20b`) restano nei file `site_sector_apply*.jsonl`, ma non sono usate nel dataset.

Provenienza: `sector_src = site_llm`, `sector_model = "openai/gpt-oss-120b (vllm local)"`, `sector_confidence`.

In [ ]:
%run site_sector_evaluate.py build/site_sector_validate_local.jsonl
%run stage5c_merge_site.py

> **Stato:** stadi 5, 5b e 6 (LLM con ricerca web) **esplorati ma non applicati**. Con il piano gratuito di Groq una chiamata con ricerca costa 4.000–230.000 token, a fronte di 8.000 token/minuto e 200.000/giorno: la validazione si è fermata a 122 eventi su 450 e l'applicazione avrebbe richiesto settimane. Codice e risultati parziali restano per trasparenza; il dataset rilasciato usa gli stadi 5c–5d.

## Stadio 5 — paese e settore mancanti con LLM

Al momento di questo esperimento restavano **6.898 paesi e 3.094 settori mancanti**. Circa metà di questi eventi non ha altro che il nome della vittima.

**Modello**: `openai/gpt-oss-120b` su Groq, con lo strumento integrato `browser_search`.

**Scelte e motivazioni**

- **Validare prima di applicare.** 450 eventi con paese *e* settore noti e unanimi, stratificati in parti uguali per tipo di input (solo nome, nome+dominio, nome+descrizione); le etichette vengono nascoste e confrontate con le risposte. Gli stessi 450 vengono classificati anche **senza ricerca**, per misurare quanto la ricerca vale.
- **Output vincolato.** Paese ISO alpha-2, settore dalle 13 classi target, confidenza, e `unknown` come risposta legittima: il prompt dice che una risposta sbagliata è peggio di un `unknown`.
- **Tracker vietati come fonte.** In un test il modello ha trovato la risposta su ransomware.live, cioè una delle fonti che aggreghiamo: in validazione ritroverebbe le etichette nascoste gonfiando l'accuratezza, in applicazione ne riprodurrebbe gli errori. Le risposte con evidenza da un tracker sono scartate.
- **Senza ricerca il modello inventa.** Nel test, senza ricerca, ha risposto con un paese sbagliato e ha citato un articolo inesistente: confidenza e URL dichiarati non sono affidabili da soli, quindi la soglia si fissa sull'accuratezza *misurata*.
- **Tutto registrato.** Ogni chiamata in `build/llm_*.jsonl` con modello, versione del prompt, data, ricerche eseguite e token. Le corse riprendono da dove si fermano.

**Vincolo pratico.** Il piano gratuito Groq concede 8.000 token al minuto; una chiamata con ricerca ne usa circa 10.000. Da qui la strategia ibrida dello stadio successivo.

```bash
export GROQ_API_KEY=...
python3 llm_enrich.py validate --n 450              # con ricerca
python3 llm_enrich.py validate --n 450 --no-search  # confronto
python3 llm_evaluate.py
```

In [ ]:
%run llm_evaluate.py

## Stadio 5b — policy: dove usare la ricerca

`llm_policy.py` → `llm_policy.json`, deciso **dai dati della validazione**, non a priori:

- **Instradamento.** Per ogni tipo di input la ricerca si usa solo se alza l'accuratezza di almeno 5 punti su almeno un attributo rispetto al modello senza ricerca. Altrove costa circa 20 volte i token senza beneficio misurato.
- **Soglie di confidenza.** Per ogni attributo e modalità, la confidenza minima tale che l'accuratezza delle risposte accettate raggiunga l'obiettivo: **90% per il paese**, **63% per il settore** — il secondo è l'accordo fra le due fonti a parità di etichetta, oltre il quale l'accuratezza non è misurabile. Se nessuna soglia raggiunge l'obiettivo, quell'attributo in quella modalità **non si applica**.

In [ ]:
%run llm_policy.py

## Stadio 6 — applicazione e integrazione

```bash
python3 llm_enrich.py apply      # instradato da llm_policy.json -> build/llm_apply.jsonl
python3 stage6_merge_llm.py      # -> build/events_final.json
```

Una risposta entra nel dataset solo se l'attributo era davvero mancante, non è `unknown`, non ha evidenza da tracker e supera la soglia della policy. Provenienza: `llm_web` (con ricerca) o `llm` (senza), più `llm_confidence`, `llm_evidence_url`, `llm_model`, `llm_prompt_version`. Chi usa il dataset può escluderle con un filtro su `country_src` / `sector_src`.

## Stadio 5e — paese e settore dalla conoscenza del modello

`llm_knowledge.py`: per gli eventi ancora senza paese o settore, quasi sempre senza dominio, `gpt-oss-120b` in locale (vllm, nessuna ricerca web) riceve il nome della vittima, il dominio se c'è e la descrizione del post sul leak site se c'è. Il nome della gang non viene passato, per non spingere il modello a ricordare i tracker. `unknown` è esplicitamente preferito a una risposta incerta. Confidenza separata per paese e settore.

**Contaminazione.** Una prima validazione su 600 eventi (200 per tipo di input) dava 86% di precisione sul paese anche con il solo nome. Diviso per data però:
- solo nome: dopo il cutoff di addestramento del modello (luglio 2024) risponde nel **9%** dei casi, contro il 77% prima;
- nome + descrizione: precisione dopo il cutoff 95,8%, cioè ragiona sul testo fornito.

Prima del cutoff il modello può ricordare le pagine dei tracker, che riportano proprio paese e settore della vittima: sono le stesse fonti, quindi la validazione sarebbe gonfiata. Le soglie sono perciò scelte **solo su eventi scoperti dopo il 1° luglio 2024** (594 eventi; `llm_knowledge_evaluate.py` → `llm_knowledge_policy.json`). Obiettivi: paese 90%, settore 63%, come per gli altri stadi.

| attributo | input | soglia | precisione | copertura |
|---|---|---|---|---|
| paese | nome + descrizione | 0,5 | 93,2% | 83% |
| paese | nome + dominio | 0,8 | 90,4% | 50% |
| paese | solo nome | — | non applicato (n=22, risponde nel 14% dei casi) | — |
| settore | nome + descrizione | 0,5 | 76,1% | 93% |
| settore | nome + dominio | 0,7 | 65,3% | 67% |
| settore | solo nome | — | non applicato | — |

**Applicazione** (`stage5e_merge_knowledge.py`, build finale): 834 paesi e 585 settori. Il paese mancante scende da 15,8% a **13,6%**, il settore da 6,6% a **5,0%**. Gli eventi con il solo nome restano mancanti per scelta.
Provenienza: `*_src = llm_knowledge`, `knowledge_input`, `knowledge_model`, `knowledge_confidence_*`.

In [ ]:
%run llm_knowledge_evaluate.py build/llm_knowledge_validate_after2024-07.jsonl llm_knowledge_policy.json
%run stage5e_merge_knowledge.py

## Precisione di ogni regola di inferenza

`validate_rules.py`: su ogni evento con valore **osservato** si finge che manchi e si applica la regola. `inherited_victim` è valutata leave-one-out: il voto usa solo gli *altri* eventi della stessa vittima, con le chiavi deduplicate per evento. Intervallo di Wilson al 95%; la baseline è la classe più frequente sullo stesso sottoinsieme.

| attributo | regola | n | precisione | IC 95% | baseline |
|---|---|---|---|---|---|
| paese | `cctld` | 7.607 | **99,3%** | 99,0–99,4 | 8,8% |
| paese | `description` | 6.470 | 95,4% | 94,9–95,9 | 38,3% |
| paese | `site_archive` / `site_live` | 272 / 35 | 96,0% / 94,3% | — | 48,2% (US) |
| paese | `llm_knowledge` (descrizione / dominio) | 250 / 135 | 93,2% / 90,4% | — | — |
| settore | `llm_knowledge` (descrizione / dominio) | 188 / 144 | 76,1% / 65,3% | — | — |
| paese | `inherited_victim` | 5.233 | 82,7% | 81,6–83,7 | 42,3% |
| paese | ~~`title_hint`~~ | 63 | 69,8% | 57,6–79,8 | 39,7% |
| settore | `site_llm` (soglia 0,90) | 164 | 82,3% | — | — |
| settore | `inherited_victim` | 6.297 | 63,8% | 62,6–64,9 | 19,7% |

Lettura:
- `title_hint` è stata **rimossa** perché sotto le altre regole (22 eventi).
- `inherited_victim` per il settore (63,8%) è allo stesso livello dell'accordo tra le fonti sullo stesso evento (59,6%, κ = 0,54): non può fare meglio delle etichette che eredita.
- Per il paese gli errori sono soprattutto multinazionali: stessa azienda, filiali attaccate in paesi diversi.
- Restringere l'eredità al solo dominio o a chiavi lunghe alza la precisione di 2–4 punti e taglia la copertura del 15–25%: non adottato, documentato.

Limite: gli eventi con valore osservato non sono un campione casuale di quelli imputati. La tabella è anche in `release/precision_by_src.json`.

In [ ]:
%run validate_rules.py

## Copertura temporale del paese: il buco 2021–2022

| anno | 2020 | 2021 | 2022 | 2023 | 2024 | 2025 | 2026 |
|---|---|---|---|---|---|---|---|
| paese mancante (observed) | 24,5% | **86,4%** | **80,5%** | 12,3% | 12,2% | 13,3% | 10,3% |
| paese mancante (enriched) | 22,9% | **67,1%** | **46,8%** | 4,3% | 3,7% | 4,9% | 5,0% |

Causa, verificata sui dati grezzi:
- **ransomfeed** copre quasi solo dal 2023 (177 record nel 2021, 603 nel 2022, 4.846 nel 2023);
- **ransomware.live** riporta il paese nel 13% dei record del 2021 e nel 20% del 2022 (contro il 93–96% dal 2024);
- **RansomLook** non ha il campo paese.

Nessun endpoint dell'API di ransomware.live (`/victim/{id}`, `/victims/search`, `/press/all`) aggiunge il paese per quegli anni: `/press/all` ha 7 voci nel 2021 e 25 nel 2022. Il buco è **strutturale nelle fonti**. Chi usa il dataset per serie temporali per paese deve limitarsi al periodo dal 2023, o trattare 2021–2022 con cautela.

In [ ]:
e = pd.read_json("release/events_enriched.json")
e["anno"] = pd.to_datetime(e.discovered).dt.year
e.groupby("anno").country.apply(lambda s: round(s.isna().mean()*100, 1))

## Audit della deduplicazione

`audit_dedup.py` cerca due tipi di errore:
- **fusione sbagliata**: record con nomi molto diversi (similarità < 0,5) nello stesso evento;
- **duplicato perso**: coppie di eventi della stessa gang entro 7 giorni con nomi simili (≥ 0,85) o stesso dominio.

L'audit ha fatto emergere tre problemi, corretti negli stadi 3–4:
1. **La chiave della vittima non puliva il nome.** `THEW ASSOCIATES HACKED. MORE THEN 50 GB…` e `THEW ASSOCIATES HACKED. MORE THEN` erano eventi diversi, così come `Ward Arcuri Foley &amp; Dwyer` e `… & Dwyer`. Ora la chiave usa la stessa pulizia del nome rilasciato (`clean_victim`: entità HTML, etichette `[..]`, code tipo "DATA LEAKED"). Eventi da 37.650 a 37.287.
2. **Chiavi solo numeriche** (`- $150.000` → `150000`) univano vittime diverse: scartate.
3. **Nomi segnaposto.** Alcune gang pubblicano `FILES`, `Breach`, `Full data dump` al posto del nome. Un evento di `clop` raccoglieva 326 record di vittime diverse. Regola guidata dai dati: un nome che nella stessa gang compare con ≥ 3 domini diversi non viene usato per collegare record (23 nomi). Evento più grande ora: 55 record, la stessa vittima pubblicata a pezzi.

Dopo le correzioni gli eventi sono 37.385. Restano 512 eventi candidati a fusione sbagliata, quasi tutti la stessa vittima con titoli diversi (`dbmgroup.com` / `\$43.6M\USA\`), e 790 coppie candidate a duplicato perso, 503 entro 3 giorni. Nel campione molte sono pubblicazioni a puntate (`Part 1` / `Part 2`) o post segnaposto (`Views: 23`).

**Da fare:** `audit/merge_candidates_sample.csv` e `audit/duplicate_candidates_sample.csv` (100 righe ciascuno, colonna `label`) vanno etichettati a mano per stimare il tasso di errore reale.

Dopo ogni nuova esecuzione degli stadi 3–4, `remap_events.py <tag>` riporta i risultati già calcolati (sito, LLM) sui nuovi `event_id`, tramite gli ID dei record grezzi.

In [ ]:
%run audit_dedup.py

## Analisi delle fonti (per il paper)

`analysis_sources.py` → `build/report_sources.json`, `figures/overlap.png`.

**Sovrapposizione.** Eventi per combinazione di fonti (RF = ransomfeed, RL = ransomware.live, LK = RansomLook):

| RF+RL+LK | LK | RL+LK | RL | RF+RL | RF+LK | RF |
|---|---|---|---|---|---|---|
| 22.103 | 4.622 | 4.428 | 2.959 | 1.878 | 855 | 540 |

**Ritardo tra tracker.** Sugli eventi visti da più fonti la mediana del ritardo è 0 giorni per ogni coppia di fonti. Stesso giorno: RF/RL 79,2%, RF/LK 75,4%, RL/LK 66,6%. Entro un giorno: 98,5%, 91,7%, 85,9%. ransomfeed è la prima fonte nel 57,6% dei casi, ransomware.live nel 25,4%, RansomLook nel 21,4% (con pari merito). Limite: le date di RansomLook per gli anni più vecchi sono in parte ricostruite a posteriori.

**Capture–recapture** (modelli log-lineari di Poisson, per anno dal 2023). Il modello con AIC minore è sempre `AB+AC`: interazione di ransomfeed con entrambe le altre fonti. Stima degli eventi che nessun tracker ha visto:

| anno | osservati | non visti (modello migliore) | IC 95% bootstrap |
|---|---|---|---|
| 2023 | 5.744 | 262 | 215–322 |
| 2024 | 6.664 | 232 | 196–273 |
| 2025 | 9.259 | 1.368 | 1.156–1.625 |
| 2026 | 8.603 | 633 | 532–747 |

Lettura: l'unione delle tre fonti copre circa l'87–97% degli eventi stimati. Le stime però **dipendono molto dal modello**: per il 2025 vanno da 42 a 1.368 non visti a seconda delle interazioni. Il modello vale come ordine di grandezza, non come conteggio. Gli altri limiti: fonti dipendenti, catturabilità diversa tra gang, e duplicati persi che gonfiano la stima.

In [ ]:
%run analysis_sources.py

## Sensibilità alla finestra temporale degli eventi

Stadio 4 rieseguito con `EVENT_WINDOW` = 1, 3, 7, 14, 30 giorni (`build/sensitivity/`, `build/report_window_sensitivity.csv`):

| finestra | eventi | 1 sola fonte | eventi 2025 | gang top (quota) | top 10 | quota US | paese mancante |
|---|---|---|---|---|---|---|---|
| 1 | 37.866 | 8.552 | 9.536 | lockbit3 (10,0%) | 41,6% | 45,8% | 17,5% |
| **3** | **37.385** | 8.121 | 9.259 | lockbit3 (10,0%) | 41,2% | 45,6% | 17,6% |
| 7 | 37.105 | 7.912 | 9.208 | lockbit3 (10,0%) | 41,2% | 45,7% | 17,6% |
| 14 | 36.736 | 7.601 | 9.135 | lockbit3 (9,8%) | 41,1% | 45,8% | 17,6% |
| 30 | 36.365 | 7.309 | 9.056 | lockbit3 (9,5%) | 40,9% | 45,9% | 17,5% |

Da 1 a 30 giorni il numero di eventi varia del 4%, e le statistiche principali (concentrazione, quota US, copertura) restano entro mezzo punto. Le conclusioni non dipendono dalla scelta di 3 giorni.

## Validazione esterna con fonti ufficiali

Le rivendicazioni dei leak site si confrontano con attacchi **confermati dalle vittime stesse**, raccolti in modo indipendente dai tracker. Abbinamento severo per nome o dominio (`ext_match.py`). Gli abbinamenti sono in `audit/*_matches.csv`, con una revisione preliminare dell'assistente (colonna `review_assistant`) **da confermare a mano**.

**1. Washington State AG** (`ext_wa.py`; data.wa.gov `sb4j-ca4h`, dati aperti): notifiche di violazione con tipo di attacco.
- 460 organizzazioni hanno notificato un attacco **ransomware** (giugno 2020 – giugno 2026). Di queste, 109 sono abbinate a un evento del dataset. Precisione dell'abbinamento alla revisione: 105 corrette, 2 sbagliate, 2 dubbie (98%). **Copertura: 22,8%.**
- Per anno: 2021 11/63, 2022 8/53, 2023 31/123, 2024 28/103, 2025 24/76.
- **Controllo negativo**: tra le notifiche di attacchi *non* ransomware (phishing, malware…) gli abbinamenti sono il 9,2%, contro il 23,7%. Il segnale è specifico del ransomware. Il residuo è in parte ransomware classificato come "malware".
- **Ritardi**: la rivendicazione arriva in mediana **43 giorni dopo l'inizio dell'attacco** (IQR 19–61), mentre la notifica ufficiale arriva in mediana **114 giorni dopo la rivendicazione** (IQR 32–206). I leak site anticipano di circa 4 mesi le fonti ufficiali.
- Lettura: circa 3 vittime confermate su 4 non compaiono su nessun leak site. È coerente con le vittime che pagano e con le gang senza leak site. La copertura va dichiarata come limite.

**2. SEC 8-K Item 1.05** (`ext_sec.py`, ricerca full-text EDGAR): incidenti informatici rilevanti delle società quotate USA, obbligatori dal dicembre 2023.
- 58 società. 16 abbinate automaticamente: alla revisione 11 sono corrette, 1 è la stessa vittima con un incidente diverso (HPE), 4 sono sbagliate. Copertura 11/58 = **19%**, un limite inferiore perché non tutti gli Item 1.05 sono ransomware.
- Campione piccolo: utile come conferma qualitativa (Prudential, Change Healthcare, Krispy Kreme, Stryker…), non come stima.

Il portale HHS (sanità, USA) non è stato usato: è un'applicazione JSF con pagina d'ingresso e non ha esportazione automatica stabile.

In [ ]:
%run ext_wa.py
%run ext_sec.py

## Etichettatura manuale

`make_label_samples.py` → `audit/label_A_merge_random.csv` (100 eventi casuali con nomi diversi), `audit/label_B_duplicates.csv` (100 coppie candidate a duplicato), `audit/label_C_imputed.csv` (270 valori imputati, 30 per origine, origine nascosta). Istruzioni in `audit/ISTRUZIONI.md`; risultati con `evaluate_labels.py` (tassi d'errore con IC di Wilson e kappa tra annotatori).

**Corrispondenza con gli standard:** `release/sector_crosswalk_naics_isic.csv` collega le 13 classi a NAICS 2022 e ISIC Rev.4. È definita sulle descrizioni delle classi, quindi è approssimata e molti-a-molti.

## Rilascio — due versioni

`make_release.py` genera entrambe dallo stesso file master (`build/events_final.json` se esiste, altrimenti `build/events_site.json`):

- **`events_observed`**: paese e settore solo se riportati dalle fonti per quell'evento (`*_src == observed`); ogni inferenza è null. Per analisi che non vogliono dipendere da alcuna imputazione.
- **`events_enriched`**: tutti i valori, con la colonna `*_src` che ne indica l'origine (`observed`, `inherited_victim`, `cctld`, `description`, `title_hint`, `site_archive`, `site_live`, `site_llm`). L'utente può filtrare le inferenze che non accetta.

Il campo `domain` è pulito solo in uscita (valori non-dominio → null): gli stadi 3–4 restano invariati, così gli `event_id` restano allineati con tutti i file intermedi.

Nel rilascio:
- `victim`: nome pulito; `victim_raw`: nome originale;
- `victim_truncated`: nome troncato dal leak site (1.006 eventi);
- `victim_placeholder`: titolo che non nomina una vittima e nessun dominio (51 eventi), tenuti perché l'attacco è stato rivendicato.

`release/README.md` è il datasheet: campi, provenienza con precisione, limiti, **termini delle fonti** ed etica.

In [ ]:
%run make_release.py

## Riepilogo (build finale)

| | `events_observed` | `events_enriched` |
|---|---|---|
| eventi / vittime / gang | 37.385 / 33.148 / 410 | uguale |
| paesi | 183 | 191 |
| paese mancante | 24,9% | **13,6%** |
| settore mancante | 14,9% | **5,0%** |

Paese mancante lungo la pipeline: stadio 4 17,6% → sito 15,8% → LLM 13,6%. Settore: 7,4% → 6,6% → 5,0%.

In [ ]:
ev = pd.read_json("build/events.json")
ev["anno"] = pd.to_datetime(ev.discovered).dt.year
print("eventi:", len(ev), "| vittime:", ev.victim.nunique(), "| gang:", ev.gang.nunique(), "| paesi:", ev.country.nunique())
print("\neventi per anno:"); print(ev.anno.value_counts().sort_index().to_string())
print("\nfonti per evento:"); print(ev.n_sources.value_counts().sort_index().to_string())
print("\nprovenienza paese:"); print(ev.country_src.value_counts().to_string())
print("\nprovenienza settore:"); print(ev.sector_src.value_counts().to_string())
print(f"\nconflitti di paese: {ev.country_conflict.sum()} ({ev.country_conflict.mean()*100:.1f}%)")

## Accordo fra fonti sul settore

Kappa di Cohen fra ransomfeed e ransomware.live sugli eventi etichettati da entrambe. **Attenzione**: l'accordo sulle etichette `empirical` è circolare — la mappatura è stata scelta proprio per massimizzarlo su questi dati. Le misure oneste sono quelle su `native` e `manual`.

In [ ]:
from sklearn.metrics import cohen_kappa_score
u = pd.read_json("build/unified.json"); m = pd.read_json("build/record_to_event.json"); u["event_id"] = m.event_id.values
rf = u[(u.source=="ransomfeed") & u.sector.notna()].groupby("event_id").agg(rf=("sector","first"), meth=("sector_map","first"))
rl = u[(u.source=="ransomlive") & u.sector.notna()].groupby("event_id").agg(rl=("sector","first"))
x = rf.join(rl, how="inner")
print(f"eventi co-etichettati: {len(x)} | accordo {(x.rf==x.rl).mean()*100:.1f}% | kappa {cohen_kappa_score(x.rf, x.rl):.3f}")
for meth, g in x.groupby("meth"):
    print(f"  {meth:9} n={len(g):6d}  accordo {(g.rf==g.rl).mean()*100:5.1f}%  kappa {cohen_kappa_score(g.rf, g.rl):.3f}")